In [5]:
# NOTE PER QUELLO CHE STAVI FACENDO PRIMA DI ANDARE A MANGIARE
# METTI PRIMA LA DEFINIZIONE DEL MODELLO E POI I DATI, GENERALIZZA IN FUNZIONI PER USARE DIVERSI DATASET
# GENERALIZZA TEST ED ESEGUI SU DATASET CON EMBEDDING E CON SIMILARITÀ

DEBUG = False

In [7]:
import torch
import pandas as pd
import networkx as nx
from torch_geometric.data import Data
from torch_geometric.utils import from_networkx
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np


def make_dataset(nodes_df:pd.DataFrame, edges_df:pd.DataFrame):
    nodes_df['GO_embeddings'] = nodes_df['GO_embeddings'].apply(lambda x: torch.tensor(eval(x), dtype=torch.float))
    nodes_df['label'] = nodes_df['label'].apply(lambda x: torch.tensor(x, dtype=torch.float))

    # Aggiungo nodi al grafo
    G = nx.Graph()
    for _, row in nodes_df.iterrows():
        num_id = int(row['num_id'])
        G.add_node(num_id, name=row['STRING_id'], x=row['GO_embeddings'], y=torch.tensor(row['label'], dtype=torch.float))

    edges_df['num_id_1'] = edges_df['num_id_1'].astype(int)
    edges_df['num_id_2'] = edges_df['num_id_2'].astype(int)
    edges = torch.tensor(edges_df[['num_id_1', 'num_id_2']].values.T, dtype=torch.int)

    # Per visualizzazione debug
    G.add_edges_from(zip(edges_df['num_id_1'], edges_df['num_id_2']))

    data = Data(
        x=torch.stack([G.nodes[n]['x'] for n in G.nodes]),  # Feature matrix
        edge_index=edges,  # Edge list
        y=torch.stack([G.nodes[n]['y'] for n in G.nodes]),
        name= [G.nodes[n]['name'] for n in G.nodes]
    )

    # Escludendo le proteine in comune estrai un 10% per testing finale
    common_proteins_mask = np.array(["_AD" in name or "_PD" in name for name in data.name])
    non_common_indices = np.where(~common_proteins_mask)[0]

    num_test = int(0.1 * len(non_common_indices))
    final_test_indices = np.random.choice(non_common_indices, size=num_test, replace=False)
    final_test_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
    final_test_mask[final_test_indices] = True

    data.final_test_mask = final_test_mask
    data.kfold_usable_mask = ~(final_test_mask)

    return data, G

nodes_emb_df = pd.read_csv("../pre_processing_output/duplicated_nodes_main_components.csv")  # Dataset con nodi (proteine)
nodes_sim_df = pd.read_csv("../pre_processing_output/duplicated_nodes_main_components_sim.csv")  # Dataset con nodi (proteine)
edges_df = pd.read_csv("../pre_processing_output/duplicated_edges_main_components.csv")  # Dataset con interazioni proteina-proteina
data_emb, _ = make_dataset(nodes_emb_df, edges_df)
data_sim, _ = make_dataset(nodes_sim_df, edges_df)

# CALCOLO STATISTICHE FEATURE

def feature_stats_and_visualization(data: Data):
    feature_values = data.x.cpu().numpy()

    # Calcolare statistiche comuni a tutte le feature
    mean_value = np.mean(feature_values)  # Media globale
    std_value = np.std(feature_values)    # Deviazione standard globale
    min_value = np.min(feature_values)    # Valore minimo globale
    max_value = np.max(feature_values)    # Valore massimo globale
    median_value = np.median(feature_values)  # Mediana globale

    global_stats = pd.DataFrame({
        "Statistic": ["Mean", "Std Dev", "Min", "Max", "Median"],
        "Value": [mean_value, std_value, min_value, max_value, median_value]
    })

    print(global_stats)

    num_features_to_plot = min(10, feature_values.shape[1])
    selected_features = np.random.choice(feature_values.shape[1], num_features_to_plot, replace=False)

    plt.figure(figsize=(12, 6))
    for feat_idx in selected_features:
        sns.kdeplot(feature_values[:, feat_idx], label=f'Feature {feat_idx}', fill=True)

    plt.xlabel("Feature Value")
    plt.ylabel("Density")
    plt.title("Distribuzione delle feature (Subset)")
    plt.legend()
    plt.show()

def graph_visualization(G: nx.Graph, data:Data):
    plt.figure(figsize=(8, 6))  # Imposta la dimensione della figura
    nx.draw(G, with_labels=True, node_size=50, font_size=8, edge_color="gray", node_color="blue")
    plt.title("Grafo costrutito da dataset")
    plt.show()

    G_temp = nx.Graph()
    G_temp.add_nodes_from(range(data.x.shape[0]))
    edges = data.edge_index.cpu().numpy()
    G_temp.add_edges_from(zip(edges[0], edges[1]))

    plt.figure(figsize=(8, 6))
    nx.draw(G_temp, with_labels=True, node_size=50, font_size=8, edge_color="gray", node_color="blue")
    plt.title("Grafo costruito da `data`")
    plt.show()

if DEBUG:
    feature_stats_and_visualization(data)
    graph_visualization(G, data)

/tmp/ipykernel_29963/3066232941.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  G.add_node(num_id, name=row['STRING_id'], x=row['GO_embeddings'], y=torch.tensor(row['label'], dtype=torch.float))
/tmp/ipykernel_29963/3066232941.py:19: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  G.add_node(num_id, name=row['STRING_id'], x=row['GO_embeddings'], y=torch.tensor(row['label'], dtype=torch.float))


In [8]:
from torch.optim import Adam
import random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


def train_once(data, lr=0.001, hidden_channels=64, out_channels=1, k_folds=5, epochs_per_fold=50):
    # Inizializzo il modello
    model = GCN(in_channels=data.x.shape[1], hidden_channels=hidden_channels, out_channels=out_channels, dropout=0)
    best_model, _ = model.k_fold_cross_validation(data, k=k_folds, epochs=epochs_per_fold, learning_rate=lr, patience=10, log=DEBUG)

    # Testing su dataset precedentemente estratto
    best_model.eval()
    with torch.no_grad():
        out = best_model(data.x, data.edge_index)
        out = torch.sigmoid(out)

        pred = (out[data.final_test_mask] > 0.5).int()
        y_true = data.y[data.final_test_mask].cpu().numpy()
        y_pred = pred.cpu().numpy()

        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)

        print("\n🧪 **Valutazione sul test set finale (10%)**")
        print(f"Accuracy: {acc:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-score: {f1:.4f}")

        plt.figure(figsize=(6, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.xlabel("Predicted")
        plt.ylabel("Actual")
        plt.title("Confusion Matrix - Final Test Set")
        plt.show()

def train_multiple_times(num_iterations: int, data, lr=0.001, hidden_channels=64, out_channels=1, k_folds=5, epochs_per_fold=50):
    from tqdm import tqdm
    from GCN import GCN as testGCN
    
    best_metrics_list = []

    with tqdm(total=num_iterations, desc="Training Progress", unit="iteration") as pbar:
        for _ in range(num_iterations):
            model = testGCN(in_channels=data.x.shape[1], hidden_channels=hidden_channels, out_channels=out_channels, dropout=0)

            _, best_metrics = model.k_fold_cross_validation(data, k=k_folds, epochs=epochs_per_fold, learning_rate=lr, patience=10)
            best_metrics_list.append(best_metrics)
            pbar.update(1)

    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    avg_metrics = {}
    std_metrics = {}
    for m in metrics:
        avg_metrics[m] = np.mean([metric[m] for metric in best_metrics_list], dtype='float64')
        std_metrics[m] = np.std([metric[m] for metric in best_metrics_list], dtype='float64')

    print('Average metrics over 100 iterations')
    for metric in metrics:
        print(f'{metric} = {avg_metrics[metric]:.5f} {std_metrics[metric]:.5f}')

train_multiple_times(num_iterations=100, data=data_emb)
train_multiple_times(num_iterations=100, data=data_sim)


Training Progress: 100%|██████████| 100/100 [01:48<00:00,  1.08s/iteration]


Average metrics over 100 iterations
Accuracy = 0.98788 0.01896
Precision = 0.98776 0.02614
Recall = 0.99060 0.02432
F1-Score = 0.98882 0.01745


Training Progress: 100%|██████████| 100/100 [02:26<00:00,  1.46s/iteration]

Average metrics over 100 iterations
Accuracy = 0.83272 0.06513
Precision = 0.90276 0.10584
Recall = 0.79555 0.09890
F1-Score = 0.83571 0.05467
